# EDA — personal_transactions_dashboard_ready.xlsx

**Objetivo:** determinar calidad, significado y suficiencia temporal para APERTURE. Análisis descriptivo, sin entrenar modelos. Los resultados textuales siguientes corresponden al archivo cuyo hash se imprime; si cambia la fuente, revisar las interpretaciones.

**Entorno:** seleccionar `.venv` del repositorio e instalar `requirements-eda.txt`. El notebook utiliza fechas originales, no fechas desplazadas para la demo.

In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from decimal import Decimal
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/eda_datasets.py').exists())
sys.path.insert(0,str(ROOT))
from scripts.eda_datasets import audit
paths=sorted((ROOT/'data/demo').rglob('personal_transactions*.xlsx'))
assert len(paths)==1, 'Archivo ausente o ambiguo'
path=paths[0]
raw=pd.read_csv(path,dtype=str) if path.suffix=='.csv' else pd.read_excel(path)
summary,missing,monthly,weekly,categories,weekly_categories=audit(path)
pio.renderers.default='plotly_mimetype'
print('Archivo:',path.name,'SHA-256:',summary['sha256'])
print('No se escribirá en el original ni en la base de la aplicación.')


Archivo: personal_transactions_dashboard_ready.xlsx SHA-256: a4c5ad85078aef4b8ce5badbf75e10e8f43c10a259dd3f44d3c19f537efd6985
No se escribirá en el original ni en la base de la aplicación.


## A. Estructura y ejemplos

**Objetivo:** conocer la unidad de registro, tipos, memoria y posibles identificadores. Los ejemplos omiten cuentas y notas para reducir exposición de información financiera.

In [2]:
print('Filas y columnas:',raw.shape)
print('Memoria profunda (bytes):',raw.memory_usage(deep=True).sum())
display(pd.DataFrame({'tipo':raw.dtypes.astype(str),'valores_distintos':raw.nunique()}))
display(raw[['Date','Category','Amount']].head())
print('Etiquetas de cuenta distintas:',summary['account_labels_count'])
print('No hay identificador explícito de persona; varias cuentas no implican varias personas.')


Filas y columnas: (806, 7)
Memoria profunda (bytes): 281717


,tipo,valores_distintos
Date,datetime64[ns],432
Description,object,65
Amount,float64,454
Transaction Type,object,2
Category,object,22
Account Name,object,3
Month,object,21


,Date,Category,Amount
0,2018-01-01,Shopping,11.11
1,2018-01-02,Mortgage & Rent,1247.44
2,2018-01-02,Restaurants,24.22
3,2018-01-03,Credit Card Payment,2298.09
4,2018-01-04,Movies & Dvds,11.76


Etiquetas de cuenta distintas: 3
No hay identificador explícito de persona; varias cuentas no implican varias personas.


**Observado:** 806 filas, 7 columnas, 3 etiquetas de cuenta; memoria profunda 281,717 bytes en el entorno auditado.

**Interpretación:** cada fila representa un movimiento según la fuente, no una observación semanal independiente.

**Implicación:** conservar cuentas y procedencia, sin construir usuarios a partir de nombres de cuenta.

## B. Calidad, faltantes y duplicados

**Objetivo:** distinguir errores confirmados, campos opcionales vacíos y señales que requieren revisión.

In [3]:
display(missing)
print({k:summary[k] for k in ['invalid_dates','invalid_amounts','nonpositive_amounts','exact_duplicate_rows','coarse_duplicate_candidates']})
print('Descripciones ambiguas candidatas:',summary['ambiguous_note_candidates'])
print('Espacios en categorías:',summary['category_whitespace_rows'])
display(px.bar(missing,x='column',y='fraction',title='Fracción faltante por columna'))


,column,missing,fraction
0,Date,0,0.0
1,Description,0,0.0
2,Amount,0,0.0
3,Transaction Type,0,0.0
4,Category,0,0.0
5,Account Name,0,0.0
6,Month,0,0.0


{'invalid_dates': 0, 'invalid_amounts': 0, 'nonpositive_amounts': 0, 'exact_duplicate_rows': 0, 'coarse_duplicate_candidates': 0}
Descripciones ambiguas candidatas: 0
Espacios en categorías: 0


**Observado:** fechas e importes válidos y positivos en las filas auditadas; cero duplicados exactos. Hay 0 filas que coinciden en una clave amplia. Columnas completamente vacías: ninguna.

**Interpretación:** una coincidencia de fecha, importe, tipo y cuenta no prueba duplicación; un importe grande puede ser legítimo.

**Implicación:** conservar filas y agregar banderas; no borrar ni imputar automáticamente.

## C. Cobertura temporal y flujos

**Objetivo:** estudiar periodos, frecuencia e ingresos/gastos registrados. Los valores se expresan en unidades menores de la unidad de análisis; no son COP salvo conversión explícita. Las semanas son lunes-domingo.

In [4]:
print('Periodo:',summary['date_min'],'a',summary['date_max'])
print('Semanas dentro de límites de fechas:',summary['calendar_bounded_weeks'])
print('Unidad de análisis:',summary['analysis_unit'])
display(monthly)
m=monthly.reset_index(names='mes');m['mes']=m.mes.astype(str)
display(px.bar(m,x='mes',y=['inflow_minor','expense_minor'],barmode='group',title='Entradas y gastos registrados por mes (unidades menores)'))
w=weekly.reset_index(names='semana');w['semana']=w.semana.astype(str)
display(px.line(w,x='semana',y='expense_minor',color='inside_date_bounds',title='Gastos semanales; bordes parciales diferenciados'))


Periodo: 2018-01-01 a 2019-09-30
Semanas dentro de límites de fechas: 91
Unidad de análisis: Unidad original; moneda no declarada en el Excel


,transactions,days_with_records,expense_minor,inflow_minor,recorded_cashflow_minor,inside_date_bounds
2018-01,32,21,206665,400000,193335,True
2018-02,33,19,208944,400000,191056,True
2018-03,43,23,217866,600000,382134,True
2018-04,48,24,286266,400000,113734,True
2018-05,40,21,1050756,400000,-650756,True
2018-06,43,22,211505,400000,188495,True
2018-07,39,22,230264,400000,169736,True
2018-08,33,19,199268,600000,400732,True
2018-09,39,19,231474,400000,168526,True
2018-10,38,20,224237,400000,175763,True


**Observado:** 2018-01-01 a 2019-09-30, 21 meses tocados y 91 semanas enteras dentro de los límites. No se observaron semanas interiores totalmente vacías.

**Interpretación:** el límite de fechas es solo una cota calendárica; no certifica que existan todas las transacciones. Ausencia de registros no equivale a gasto cero.

**Implicación:** revisar bordes y cobertura antes del experimento; no desplazar fechas para evaluar ni consultar tasas.

## D. Categorías, dispersión y valores extremos

**Objetivo:** estudiar frecuencia, importe, media/mediana y actividad semanal, además de señales IQR. Los valores extremos se conservan.

In [5]:
display(pd.Series(summary['category_counts'],name='frecuencia').to_frame())
display(categories)
top=list(categories.head(5).category)
z=weekly_categories[top].reset_index(names='semana');z['semana']=z.semana.astype(str)
display(px.line(z,x='semana',y=top,title='Evolución por categoría (unidades menores)'))
amount_col='INR' if 'INR' in raw else 'Amount'
x=pd.to_numeric(raw[amount_col],errors='coerce')
display(px.histogram(x=x,nbins=35,title='Distribución de importes: gráfico exploratorio, sin eliminar extremos'))


,frecuencia
Credit Card Payment,143
Groceries,105
Restaurants,81
Utilities,63
Shopping,60
Gas & Fuel,52
Paycheck,46
Home Improvement,36
Coffee Shops,31
Alcohol & Bars,25


,category,transactions,total_minor,median_transaction_minor,mean_transaction_decimal,active_bounded_weeks,fraction_zero_weeks,iqr_upper_flags
13,Mortgage & Rent,21,2475450,120918,1178.785714285714285714285714,21,0.769231,0.0
10,Home Improvement,36,1909287,3430.5,530.3575,25,0.725275,5.0
8,Groceries,105,279521,1577,26.62104761904761904761904762,66,0.274725,9.0
19,Utilities,63,277600,3500,44.06349206349206349206349206,47,0.483516,0.0
16,Restaurants,81,261302,2585,32.25950617283950617283950617,52,0.428571,5.0
17,Shopping,60,197324,1846,32.88733333333333333333333333,47,0.483516,4.0
7,Gas & Fuel,52,171517,3467.5,32.98403846153846153846153846,47,0.483516,0.0
12,Mobile Phone,21,168040,8940,80.01904761904761904761904762,21,0.769231,0.0
11,Internet,21,157088,7499,74.80380952380952380952380952,20,0.780220,1.0
1,Auto Insurance,18,135000,7500,75,18,0.802198,0.0


**Observado:** las categorías tienen frecuencias desiguales; véanse `active_bounded_weeks` y `fraction_zero_weeks`. El criterio IQR se aplica solo a grupos con al menos ocho registros.

**Interpretación:** categorías mensuales como vivienda pueden contener muchos ceros semanales. La media y la mediana describen aspectos diferentes de una distribución sesgada.

**Implicación:** seleccionar categorías con suficiente actividad únicamente dentro de entrenamiento; no confundir una bandera IQR con error o fraude.

## E. Limpieza semántica y monedas

**Objetivo:** documentar las transformaciones permitidas y las decisiones pendientes.

In [6]:
print('Monedas:',summary['currency_counts'])
print('Pagos de tarjeta:',summary['card_payment_rows'])
print('Patas sin pareja con coincidencia estricta:',summary['unpaired_card_legs_coarse_match'])
print('Diferencias entre Amount e INR:',summary['original_vs_inr_differences'])
print('Account.1 igual a Amount:',summary['amount_account1_equal'])


Monedas: {'NO_DECLARADA': 806}
Pagos de tarjeta: 143
Patas sin pareja con coincidencia estricta: 97
Diferencias entre Amount e INR: None
Account.1 igual a Amount: None


**Observado:** 143 pagos de tarjeta, 72 créditos y 71 débitos. La moneda no aparece en las columnas, en el formato numérico General ni en la descripción oficial consultada. Month concuerda con Date.

**Interpretación:** pagos de tarjeta no son consumo adicional. El emparejamiento por mismo día e importe no permite concluir que falten movimientos. USD permanece sin verificar.

**Implicación:** conservar originales y marcar conciliación; la preparación deja moneda y COP nulos hasta tener evidencia de la moneda.

## F. Dictamen y reproducibilidad

**Objetivo:** decidir usos viables sin inventar resultados predictivos.

In [7]:
print('El original permanece intacto:',hashlib.sha256(path.read_bytes()).hexdigest()==summary['sha256'])
print('Modelos entrenados en este notebook: ninguno')


El original permanece intacto: True
Modelos entrenados en este notebook: ninguno


**Dictamen:** Candidato principal para un experimento temporal exploratorio, tras verificar moneda y conciliación. Un MLP no está justificado por el número de transacciones: después de rezagos y prueba temporal hay muchas menos observaciones.

**Siguiente paso:** revisión humana de tipos dudosos, confirmación de cobertura y validación temporal frente a última semana/promedio móvil. Los sintéticos se reservan para desarrollo y pruebas.

Fuentes: [Kaggle Personal Finance](https://www.kaggle.com/datasets/entrepreneurlife/personal-finance/data), [Kaggle My Expenses](https://www.kaggle.com/datasets/tharunprabu/my-expenses-data), [Frankfurter](https://frankfurter.dev/).